# 03. Deep Sequence Models on Chosen Series

This notebook reuses the representative `chosen` selection from `01_eda-2.ipynb`, then trains target-only recurrent neural networks on those four series:

- vanilla RNN
- GRU
- LSTM

Each chosen series gets an adaptive chronological split: the first 80% of observations are used for training and the last 20% for validation. This avoids the `VAL_CUTOFF=2880` issue where two selected high-weight/stable series have only seven pre-cutoff observations.


## 0. Environment Check

Run this first. The local `af` virtual environment and the Homebrew Jupyter kernel may not have a working `torch`/parquet stack, so this cell prints the active interpreter and imports the packages this notebook needs.


In [1]:
import os
import sys
import importlib
from pathlib import Path

# Keep matplotlib/font caches out of unwritable home directories when possible.
os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'matplotlib-cache'))
os.environ.setdefault('XDG_CACHE_HOME', str(Path('/tmp') / 'xdg-cache'))

print('Python executable:', sys.executable)
print('Python version:   ', sys.version.replace('\n', ' '))

required_modules = ['pandas', 'pyarrow', 'numpy', 'matplotlib', 'sklearn', 'torch']
versions = {}
missing = []
for module_name in required_modules:
    try:
        module = importlib.import_module(module_name)
        versions[module_name] = getattr(module, '__version__', 'ok')
    except Exception as exc:
        versions[module_name] = f'MISSING: {type(exc).__name__}: {exc}'
        missing.append(module_name)

for module_name, version in versions.items():
    print(f'{module_name:>10}: {version}')

if missing:
    raise ImportError(
        'This notebook needs the missing modules above. In this workspace the known-good '
        'interpreter is /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.'
    )


Python executable: /Users/kaushalnandaniya/Desktop/MnC/sem-6/AF/tut/af/bin/python
Python version:    3.12.4 (v3.12.4:8e8a4baf65, Jun  6 2024, 17:33:18) [Clang 13.0.0 (clang-1300.0.29.30)]
    pandas: 3.0.0
   pyarrow: MISSING: ModuleNotFoundError: No module named 'pyarrow'
     numpy: 2.4.2
matplotlib: 3.10.8
   sklearn: 1.8.0
     torch: MISSING: ImportError: dlopen(/Users/kaushalnandaniya/Desktop/MnC/sem-6/AF/tut/af/lib/python3.12/site-packages/torch/_C.cpython-312-darwin.so, 0x0002): Library not loaded: @rpath/libtorch_cpu.dylib
  Referenced from: <13B7189A-A13E-36CA-A6F2-F8202B247212> /Users/kaushalnandaniya/Desktop/MnC/sem-6/AF/tut/af/lib/python3.12/site-packages/torch/lib/libtorch_python.dylib
  Reason: tried: '/Users/kaushalnandaniya/Desktop/MnC/sem-6/AF/tut/af/lib/python3.12/site-packages/torch/lib/libtorch_cpu.dylib' (no such file), '/Users/kaushalnandaniya/Desktop/MnC/sem-6/AF/tut/af/lib/python3.12/site-packages/torch/lib/libtorch_cpu.dylib' (no such file), '/Users/kaushalnan

ImportError: This notebook needs the missing modules above. In this workspace the known-good interpreter is /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.

## 1. Setup and Data Load

Only the columns needed for target-only sequence modeling are loaded from `train.parquet`.


In [ ]:
import warnings
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'ts-forecasting'
VAL_CUTOFF = 2880
SEED = 42

series_keys = ['code', 'sub_code', 'sub_category', 'horizon']
required_columns = [*series_keys, 'ts_index', 'y_target', 'weight']

# Modeling hyperparameters. Set NOTEBOOK_SMOKE=1 in the environment for a quick one-series run.
LOOKBACK = 24
TRAIN_FRACTION = 0.80
HIDDEN_SIZE = 32
NUM_LAYERS = 1
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_EPOCHS = 100
PATIENCE = 12
ARCHITECTURES = ['RNN', 'GRU', 'LSTM']
SMOKE_TEST = os.environ.get('NOTEBOOK_SMOKE', '0') == '1'

if SMOKE_TEST:
    MAX_EPOCHS = 3
    PATIENCE = 1
    ARCHITECTURES = ['GRU']
    print('SMOKE_TEST enabled: running one chosen series and one GRU for a few epochs.')

DEVICE = torch.device('cpu')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:    ', DATA_DIR)
print('Device:      ', DEVICE)


In [ ]:
def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_all_seeds(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

train = pd.read_parquet(DATA_DIR / 'train.parquet', columns=required_columns)
train = train.sort_values(series_keys + ['ts_index']).reset_index(drop=True)

print(f'Train shape: {train.shape}')
print(f'Train ts_index range: {int(train.ts_index.min())} to {int(train.ts_index.max())}')
print(f'Loaded columns: {required_columns}')
train.head()


## 2. Recreate `chosen`

This is the same representative-series logic from the EDA notebook: longest history, highest total weight, most volatile, and most stable among eligible series.


In [ ]:
series_stats = (
    train.groupby(series_keys)
         .agg(length=('ts_index', 'size'),
              start=('ts_index', 'min'),
              end=('ts_index', 'max'),
              total_weight=('weight', 'sum'),
              target_std=('y_target', 'std'))
         .reset_index()
)
series_stats['crosses_cutoff'] = (series_stats['start'] <= VAL_CUTOFF) & (series_stats['end'] > VAL_CUTOFF)

eligible = series_stats[(series_stats['crosses_cutoff']) & (series_stats['length'] >= 120)].copy()
eligible['target_std'] = eligible['target_std'].fillna(0.0)
stable_pool = eligible[eligible['target_std'] > 0]

chosen = pd.concat([
    eligible.nlargest(1, 'length').assign(reason='longest history'),
    eligible.nlargest(1, 'total_weight').assign(reason='highest total weight'),
    eligible.nlargest(1, 'target_std').assign(reason='most volatile'),
    stable_pool.nsmallest(1, 'target_std').assign(reason='most stable'),
], ignore_index=True).drop_duplicates(subset=series_keys)

print(f'Total series: {len(series_stats):,}')
print(f'Eligible series: {len(eligible):,}')
chosen[[*series_keys, 'reason', 'length', 'start', 'end', 'total_weight', 'target_std']]


In [ ]:
def series_label(row):
    return f"{row['code']}_{row['sub_code']}_{row['sub_category']}_H{int(row['horizon'])}"

chosen = chosen.copy()
chosen['series'] = chosen.apply(series_label, axis=1)

chosen[[
    'series', *series_keys, 'reason', 'length', 'start', 'end', 'total_weight', 'target_std'
]].sort_values('reason')


## 3. Metrics and Series Utilities

The score functions mirror the classical notebook: weighted skill score, weighted RMSE, weighted MAE, and MASE.


In [ ]:
def weighted_skill(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    denom = np.sum(weight * (y_true ** 2))
    if denom == 0:
        return np.nan
    ratio = np.sum(weight * ((y_true - y_pred) ** 2)) / denom
    ratio = min(max(float(ratio), 0.0), 1.0)
    return float(np.sqrt(1.0 - ratio))


def weighted_rmse(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    weight_sum = np.sum(weight)
    if weight_sum == 0:
        return np.nan
    return float(np.sqrt(np.sum(weight * ((y_true - y_pred) ** 2)) / weight_sum))


def weighted_mae(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weight = np.asarray(weight, dtype=float)
    weight_sum = np.sum(weight)
    if weight_sum == 0:
        return np.nan
    return float(np.sum(weight * np.abs(y_true - y_pred)) / weight_sum)


def mase(y_true, y_pred, train_scale):
    if not np.isfinite(train_scale) or train_scale == 0:
        return np.nan
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)) / train_scale)


def metric_dict(y_true, y_pred, weight, train_scale):
    return {
        'skill_score': weighted_skill(y_true, y_pred, weight),
        'weighted_rmse': weighted_rmse(y_true, y_pred, weight),
        'weighted_mae': weighted_mae(y_true, y_pred, weight),
        'mase': mase(y_true, y_pred, train_scale),
    }


In [ ]:
def get_series_df(meta_row):
    mask = np.ones(len(train), dtype=bool)
    for key in series_keys:
        mask &= train[key].eq(meta_row[key]).to_numpy()
    return train.loc[mask, ['ts_index', 'y_target', 'weight']].sort_values('ts_index').reset_index(drop=True)


def adaptive_split(series_df, train_fraction=TRAIN_FRACTION, lookback=LOOKBACK):
    n = len(series_df)
    split_idx = int(np.floor(n * train_fraction))
    split_idx = max(split_idx, lookback + 1)
    split_idx = min(split_idx, n - 1)
    train_part = series_df.iloc[:split_idx].copy()
    val_part = series_df.iloc[split_idx:].copy()
    return train_part, val_part


split_rows = []
for _, row in chosen.iterrows():
    series_df = get_series_df(row)
    train_part, val_part = adaptive_split(series_df)
    split_rows.append({
        'series': row['series'],
        'reason': row['reason'],
        'length': len(series_df),
        'train_len': len(train_part),
        'val_len': len(val_part),
        'train_ts_start': int(train_part['ts_index'].min()),
        'train_ts_end': int(train_part['ts_index'].max()),
        'val_ts_start': int(val_part['ts_index'].min()),
        'val_ts_end': int(val_part['ts_index'].max()),
    })

split_summary = pd.DataFrame(split_rows)
split_summary


## 4. Windowing and Model Definition

Training uses one-step supervised windows. Validation forecasts are recursive/open-loop: after each predicted value, the prediction is appended back into the lookback context.


In [ ]:
def make_windows(values, lookback=LOOKBACK):
    values = np.asarray(values, dtype=np.float32)
    if len(values) <= lookback:
        return (
            np.empty((0, lookback, 1), dtype=np.float32),
            np.empty((0,), dtype=np.float32),
        )
    X, y = [], []
    for start in range(len(values) - lookback):
        end = start + lookback
        X.append(values[start:end])
        y.append(values[end])
    X = np.asarray(X, dtype=np.float32).reshape(-1, lookback, 1)
    y = np.asarray(y, dtype=np.float32)
    return X, y


def make_validation_windows(train_scaled, val_scaled, lookback=LOOKBACK):
    train_scaled = np.asarray(train_scaled, dtype=np.float32)
    val_scaled = np.asarray(val_scaled, dtype=np.float32)
    context = np.concatenate([train_scaled[-lookback:], val_scaled])
    X, y = [], []
    for i in range(len(val_scaled)):
        X.append(context[i:i + lookback])
        y.append(val_scaled[i])
    X = np.asarray(X, dtype=np.float32).reshape(-1, lookback, 1)
    y = np.asarray(y, dtype=np.float32)
    return X, y


class SequenceRegressor(nn.Module):
    def __init__(self, model_kind, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        model_kind = model_kind.upper()
        self.model_kind = model_kind
        if model_kind == 'RNN':
            self.recurrent = nn.RNN(
                input_size=1,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                nonlinearity='tanh',
            )
        elif model_kind == 'GRU':
            self.recurrent = nn.GRU(
                input_size=1,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
            )
        elif model_kind == 'LSTM':
            self.recurrent = nn.LSTM(
                input_size=1,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
            )
        else:
            raise ValueError(f'Unknown model_kind: {model_kind}')
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.recurrent(x)
        return self.head(out[:, -1, :]).squeeze(-1)


In [ ]:
def recursive_forecast(model, train_scaled, steps, mean, std, lookback=LOOKBACK, device=DEVICE):
    model.eval()
    context = list(np.asarray(train_scaled[-lookback:], dtype=np.float32))
    preds_scaled = []
    with torch.no_grad():
        for _ in range(steps):
            x = torch.tensor(context[-lookback:], dtype=torch.float32, device=device).view(1, lookback, 1)
            pred_scaled = float(model(x).detach().cpu().numpy()[0])
            preds_scaled.append(pred_scaled)
            context.append(pred_scaled)
    preds_scaled = np.asarray(preds_scaled, dtype=float)
    return preds_scaled * std + mean


def train_sequence_model(y_train, y_val, model_kind, seed=SEED):
    set_all_seeds(seed)

    y_train = np.asarray(y_train, dtype=float)
    y_val = np.asarray(y_val, dtype=float)
    mean = float(np.mean(y_train))
    std = float(np.std(y_train))
    if not np.isfinite(std) or std == 0:
        std = 1.0

    train_scaled = (y_train - mean) / std
    val_scaled = (y_val - mean) / std

    X_train, target_train = make_windows(train_scaled, LOOKBACK)
    X_val, target_val = make_validation_windows(train_scaled, val_scaled, LOOKBACK)
    if len(X_train) == 0:
        raise ValueError(f'not enough training points for LOOKBACK={LOOKBACK}')

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(target_train))
    train_loader = DataLoader(
        train_ds,
        batch_size=min(BATCH_SIZE, len(train_ds)),
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    X_val_t = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    target_val_t = torch.tensor(target_val, dtype=torch.float32, device=DEVICE)

    model = SequenceRegressor(model_kind).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()

    best_state = None
    best_val_loss = np.inf
    bad_epochs = 0
    history_rows = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        batch_losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu().item()))

        train_loss = float(np.mean(batch_losses))
        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(X_val_t), target_val_t).detach().cpu().item())

        history_rows.append({
            'epoch': epoch,
            'model': model_kind,
            'train_loss': train_loss,
            'val_loss': val_loss,
        })

        if val_loss < best_val_loss - 1e-8:
            best_val_loss = val_loss
            best_state = {name: param.detach().cpu().clone() for name, param in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    pred = recursive_forecast(model, train_scaled, steps=len(y_val), mean=mean, std=std)
    history = pd.DataFrame(history_rows)
    return pred, history


## 5. Train and Evaluate

The baseline is naive persistence: repeat the last training value across the validation horizon.


In [ ]:
experiment_series = chosen.head(1).copy() if SMOKE_TEST else chosen.copy()

results_rows = []
forecast_store = {}
history_store = {}

def add_result(meta_row, model_name, train_part, val_part, pred, status='ok', error=None):
    y_val = val_part['y_target'].to_numpy(dtype=float)
    w_val = val_part['weight'].to_numpy(dtype=float)
    y_train = train_part['y_target'].to_numpy(dtype=float)
    train_scale = float(np.mean(np.abs(np.diff(y_train)))) if len(y_train) > 1 else np.nan

    row = {
        'series': meta_row['series'],
        'reason': meta_row['reason'],
        'code': meta_row['code'],
        'sub_code': meta_row['sub_code'],
        'sub_category': meta_row['sub_category'],
        'horizon': int(meta_row['horizon']),
        'model': model_name,
        'train_len': int(len(train_part)),
        'val_len': int(len(val_part)),
        'status': status,
        'error': error,
        'skill_score': np.nan,
        'weighted_rmse': np.nan,
        'weighted_mae': np.nan,
        'mase': np.nan,
    }
    if status == 'ok' and pred is not None:
        row.update(metric_dict(y_val, pred, w_val, train_scale))
    results_rows.append(row)

for series_idx, (_, meta_row) in enumerate(experiment_series.iterrows(), start=1):
    series_df = get_series_df(meta_row)
    train_part, val_part = adaptive_split(series_df)
    series_id = meta_row['series']
    print(f"[{series_idx}/{len(experiment_series)}] {series_id} ({meta_row['reason']}): train={len(train_part)}, val={len(val_part)}")

    y_train = train_part['y_target'].to_numpy(dtype=float)
    y_val = val_part['y_target'].to_numpy(dtype=float)

    naive_pred = np.repeat(float(y_train[-1]), len(y_val))
    forecast_store[(series_id, 'Naive')] = naive_pred
    add_result(meta_row, 'Naive', train_part, val_part, naive_pred)

    for model_kind in ARCHITECTURES:
        try:
            pred, history = train_sequence_model(y_train, y_val, model_kind, seed=SEED)
            if len(pred) != len(y_val):
                raise ValueError(f'forecast length {len(pred)} != validation length {len(y_val)}')
            if not np.all(np.isfinite(pred)):
                raise ValueError('forecast contains non-finite values')
            forecast_store[(series_id, model_kind)] = pred
            history = history.copy()
            history['series'] = series_id
            history['reason'] = meta_row['reason']
            history_store[(series_id, model_kind)] = history
            add_result(meta_row, model_kind, train_part, val_part, pred)
            print(f'  {model_kind}: ok, epochs={len(history)}, best_val_loss={history.val_loss.min():.5f}')
        except Exception as exc:
            add_result(meta_row, model_kind, train_part, val_part, None, status='failed', error=f'{type(exc).__name__}: {exc}')
            print(f'  {model_kind}: failed ({type(exc).__name__}: {exc})')

results = pd.DataFrame(results_rows)
results.sort_values(['series', 'model']).reset_index(drop=True)


## 6. Leaderboard

Skill is higher-is-better. RMSE, MAE, and MASE are lower-is-better.


In [ ]:
ok = results[results['status'] == 'ok'].copy()
leaderboard = (
    ok.groupby('model')
      .agg(
          n_series=('series', 'nunique'),
          mean_skill=('skill_score', 'mean'),
          median_skill=('skill_score', 'median'),
          mean_weighted_rmse=('weighted_rmse', 'mean'),
          mean_weighted_mae=('weighted_mae', 'mean'),
          mean_mase=('mase', 'mean'),
      )
      .sort_values(['mean_skill', 'median_skill'], ascending=False)
)
leaderboard


In [ ]:
per_series_winner = (
    ok.loc[ok.groupby('series')['skill_score'].idxmax(),
           ['series', 'reason', 'model', 'skill_score', 'weighted_rmse', 'weighted_mae', 'mase']]
      .sort_values('reason')
      .reset_index(drop=True)
)
per_series_winner


## 7. Forecast Plots

Each panel shows the adaptive train/validation split, validation truth, naive baseline, and recurrent model forecasts.


In [ ]:
plot_models = ['Naive', *ARCHITECTURES]
colors = {
    'Naive': '#4C72B0',
    'RNN': '#55A868',
    'GRU': '#C44E52',
    'LSTM': '#8172B3',
}

fig, axes = plt.subplots(len(experiment_series), 1, figsize=(13, 3.3 * len(experiment_series)), sharex=False)
axes = np.atleast_1d(axes)

for ax, (_, meta_row) in zip(axes, experiment_series.iterrows()):
    series_df = get_series_df(meta_row)
    train_part, val_part = adaptive_split(series_df)
    series_id = meta_row['series']

    ax.plot(train_part['ts_index'], train_part['y_target'], color='black', linewidth=0.9, alpha=0.75, label='train')
    ax.plot(val_part['ts_index'], val_part['y_target'], color='#DD8452', linewidth=1.4, label='validation truth')
    ax.axvline(val_part['ts_index'].iloc[0], color='black', linestyle='--', linewidth=0.8, alpha=0.65)

    for model_name in plot_models:
        pred = forecast_store.get((series_id, model_name))
        if pred is None:
            continue
        ax.plot(
            val_part['ts_index'],
            pred,
            linewidth=1.1,
            alpha=0.9,
            color=colors.get(model_name),
            label=f'{model_name} forecast',
        )

    ax.set_title(f"{meta_row['reason']}: {series_id}")
    ax.set_xlabel('ts_index')
    ax.set_ylabel('y_target')
    ax.legend(fontsize=8, loc='best', ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Training Curves

These curves use teacher-forced one-step validation windows for early stopping diagnostics. The scored forecasts above are still recursive/open-loop.


In [ ]:
if history_store:
    history_df = pd.concat(history_store.values(), ignore_index=True)
else:
    history_df = pd.DataFrame(columns=['epoch', 'model', 'train_loss', 'val_loss', 'series', 'reason'])

history_df.head()


In [ ]:
if history_df.empty:
    print('No neural model histories to plot.')
else:
    fig, axes = plt.subplots(len(experiment_series), 1, figsize=(13, 3.2 * len(experiment_series)), sharex=False)
    axes = np.atleast_1d(axes)

    for ax, (_, meta_row) in zip(axes, experiment_series.iterrows()):
        series_id = meta_row['series']
        sub = history_df[history_df['series'] == series_id]
        for model_name in ARCHITECTURES:
            h = sub[sub['model'] == model_name]
            if h.empty:
                continue
            ax.plot(h['epoch'], h['train_loss'], linestyle='--', linewidth=1.0, label=f'{model_name} train')
            ax.plot(h['epoch'], h['val_loss'], linewidth=1.3, label=f'{model_name} val')
        ax.set_title(f"Loss curves: {meta_row['reason']} ({series_id})")
        ax.set_xlabel('epoch')
        ax.set_ylabel('MSE on standardized target')
        ax.legend(fontsize=8, loc='best', ncol=3)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## 9. Failure Check and Takeaways

Use this final cell to see whether any model failed and to summarize the ranking.


In [ ]:
status_table = results.groupby(['model', 'status']).size().unstack(fill_value=0)
print('Status table:')
print(status_table)

if (results['status'] != 'ok').any():
    print('\nFailures:')
    display(results[results['status'] != 'ok'][['series', 'model', 'error']])

print('\nLeaderboard:')
display(leaderboard)

print('\nPer-series winners:')
display(per_series_winner)
